In [1]:
#!/usr/bin/env python3
"""
de_poisson_application.py
=========================

Zastosowanie modeli dyfuzyjnych (SDEdit + FunDPS) do rozwiązywania
1D równania Poissona jako zadania numerycznego.

══════════════════════════════════════════════════════════════════════
RÓWNANIE DOCELOWE:
  −u″(x) = f(x),  x ∈ [0,1],  u(0) = u(1) = 0

  Rodzina parametryczna (N trybów Fouriera):
    f(x) = Σₙ cₙ · sin(nπx),  cₙ ~ N(0, 1/n²)
    u(x) = Σₙ cₙ / (nπ)² · sin(nπx)    ← rozwiązanie analityczne

  Dlaczego Poisson 1D?
    • Jeden z 5 równań benchmarkowych FunDPS (Yao et al., NeurIPS 2025)
    • Rozwiązania to funkcje gładkie ~ training data (sinc, mixed_freq, chirp)
    • Eksakt analytyczny → walidacja metryczna
    • Naturalne interpretacje obu metod:
        SDEdit  → korektor błędu różnic skończonych (numeryczny szum Eulera)
        FunDPS  → rekonstrukcja z 10% pomiarów czujnikowych (problem odwrotny)

══════════════════════════════════════════════════════════════════════
EKSPERYMENTY:

  A. SDEdit jako korektor numeryczny
     • Wejście:  rozwiązanie MRS zaburzone szumem implementacyjnym
     • Model:    Conv1D, T=80, harmonogram liniowy (optymalne z Eksp. II)
     • Wyjście:  odszumione rozwiązanie BVP
     • Metryka:  redukcja MSE względem MRS bazowego

  B. FunDPS jako solver problemów odwrotnych
     • Wejście:  13 / 128 punktów pomiarowych (10%) + warunki brzegowe
     • Model:    EDM MLP, biały szum, ζ=3.0, steps=100 (optymalne z Eksp. III)
     • Wyjście:  zrekonstruowana pełna trajektoria u(x)
     • Metryka:  błąd L₂ relative i MSE

══════════════════════════════════════════════════════════════════════
Odniesienia:
  • FunDPS: Yao et al. "Guided Diffusion Sampling on Function Spaces
            with Applications to PDEs" - NeurIPS 2025
  • SDEdit: Meng et al. "SDEdit: Guided Image Synthesis and Editing
            with Stochastic Differential Equations" - ICLR 2022
  • DPS:    Chung et al. "Diffusion Posterior Sampling for General
            Noisy Inverse Problems" - ICLR 2023
══════════════════════════════════════════════════════════════════════
"""

import os
import sys
import time
import pickle
import warnings
import math

import numpy as np
from scipy.linalg import solve as scipy_solve

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm import tqdm

warnings.filterwarnings('ignore')

try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _HERE = os.getcwd()

_PARENT = os.path.abspath(os.path.join(_HERE, ".."))

sys.path.insert(0, _HERE)
sys.path.insert(0, _PARENT)

try:
    from models.ddpm1d import DDPM1D, SinusoidalPositionEmbeddings, get_beta_schedule
    from models.conv1d  import DenoiseNet1D_Conv
    from models.unet    import DenoiseNet1D_UNet
    from models.mlp     import DenoiseNet1D_MLP
    from models.edm1d   import EDMDenoiser1D, FunDPSSampler, ForwardOperator
    _MODULES_OK = True
    print("[OK] Moduły projektu wczytane pomyślnie z katalogu głównego.")
except ImportError as exc:
    _MODULES_OK = False
    print(f"[WARN] Nie można zaimportować modułów projektu: {exc}")
try:
    from models.ddpm1d import DDPM1D, SinusoidalPositionEmbeddings, get_beta_schedule
    from models.conv1d  import DenoiseNet1D_Conv
    from models.unet    import DenoiseNet1D_UNet
    from models.mlp     import DenoiseNet1D_MLP
    from models.edm1d   import EDMDenoiser1D, FunDPSSampler, ForwardOperator
    _MODULES_OK = True
    print("[OK] Moduły projektu wczytane poprawnie.")
except ImportError as exc:
    _MODULES_OK = False
    print(f"[WARN] Nie można zaimportować modułów projektu: {exc}")
    print("       Używam wbudowanych stub-klas do demonstracji.")



DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

N_POINTS  = 128         
N_MODES   = 8            
N_TRAIN   = 5000
N_VAL     = 500
N_TEST    = 100


FUNDPS_ZETA         = 2.0    
FUNDPS_STEPS        = 5     
FUNDPS_PRIOR_EPOCHS = 2000   
FUNDPS_OBS_RATIO    = 0.10   
FUNDPS_LR           = 1e-3

SDEDIT_T           = 80
SDEDIT_SCHEDULE    = 'linear'
SDEDIT_T_RATIO     = 0.20      
SDEDIT_SKIP        = 2          
SDEDIT_LR          = 1e-3
SDEDIT_BATCH       = 128
SDEDIT_EPOCHS      = 5000       
SDEDIT_PATIENCE    = 200

AMP_DTYPE = (torch.bfloat16
             if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
             else torch.float16)

OUT_DIR = 'experiments/de_application'
os.makedirs(OUT_DIR, exist_ok=True)
print(f"[INFO] Wyniki będą zapisane w: {OUT_DIR}/")
print(f"[INFO] Urządzenie obliczeniowe: {DEVICE}\n")



class PoissonBVPGenerator:
    """
    Generator rozwiązań analitycznych i numerycznych dla 1D równania Poissona:

        −u″(x) = f(x),  x ∈ [0,1],  u(0) = u(1) = 0

    Rodzina parametryczna:
        f(x) = Σₙ₌₁ᴺ cₙ · sin(nπx)
        u(x) = Σₙ₌₁ᴺ cₙ / (nπ)² · sin(nπx)    ← dokładne rozwiązanie

    Amplitudy cₙ ~ N(0, σₙ²) z σₙ = 1/n (zanikające dla wyższych trybów)
    """

    def __init__(self, n_points: int = N_POINTS,
                 n_modes:  int = N_MODES,
                 seed:     int = 42):
        self.n_points = n_points
        self.n_modes  = n_modes
        self.x        = np.linspace(0.0, 1.0, n_points)  
        self.h        = self.x[1] - self.x[0]            
        self.rng      = np.random.default_rng(seed)

        n_idx         = np.arange(1, n_modes + 1, dtype=float)
        self._sin_basis = np.sin(
            np.outer(n_idx, np.pi * self.x)   
        )  # sin(n·π·x) dla n=1..N

    # ---------------------------------------------------------------- #
    # Dokładne rozwiązanie                                              #
    # ---------------------------------------------------------------- #
    def analytical_solution(self, coeffs: np.ndarray = None):
        """
        Zwraca (f, u_exact) dla zadanych współczynników Fouriera.

        f(x) = Σ cₙ sin(nπx)
        u(x) = Σ cₙ/(nπ)² sin(nπx)

        Parametry
        ----------
        coeffs : (n_modes,) ndarray lub None
            Amplitudy harmonicznych. Jeśli None – próbkowane losowo.

        Zwraca
        ------
        f : (n_points,) ndarray  – człon źródłowy
        u : (n_points,) ndarray  – rozwiązanie analityczne
        """
        if coeffs is None:
            # cₙ ~ N(0, 1/n²) – szybko zanikające amplitudy
            n_idx  = np.arange(1, self.n_modes + 1, dtype=float)
            coeffs = self.rng.standard_normal(self.n_modes) / n_idx

        c = coeffs[:, None]                      # (n_modes, 1)
        f = np.sum(c * self._sin_basis, axis=0)  # (n_points,)

        n_idx    = np.arange(1, self.n_modes + 1, dtype=float)
        lambda_n = (n_idx * np.pi) ** 2          # (nπ)²
        u = np.sum((c / lambda_n[:, None]) * self._sin_basis, axis=0)

        return f.astype(np.float32), u.astype(np.float32)

    # ---------------------------------------------------------------- #
    # Rozwiązanie MRS + szum (różnice skończone)  #
    # ---------------------------------------------------------------- #
    def fd_solution(self, f: np.ndarray, noise_std: float = 0.0) -> np.ndarray:
        """
        Rozwiązuje MRS + szum −u″ = f na siatce jednorodnej z krokiem h.
        Opcjonalnie dodaje szum do prawej strony układu.

        Macierz różnic skończonych (tridiagonalna, N−2 × N−2):
            A = (1/h²)·tridiag(−1, 2, −1)

        Schemat 2. rzędu (O(h²)), lecz zaburzony szumem
        symuluje błędy implementacyjne / szum pomiarowy.
        """
        N  = self.n_points
        h2 = self.h ** 2

        diag  = 2.0 * np.ones(N - 2)
        off   = -1.0 * np.ones(N - 3)
        A     = np.diag(diag) + np.diag(off, 1) + np.diag(off, -1)
        A    /= h2

        rhs  = f[1:-1].copy()

        if noise_std > 0.0:
            rhs += noise_std * self.rng.standard_normal(N - 2)

        try:
            u_int = scipy_solve(A, rhs)
        except Exception:
            u_int = np.linalg.solve(A, rhs)

        # warunki brzegowe u(0)=u(1)=0
        u_fd = np.zeros(N, dtype=np.float32)
        u_fd[1:-1] = u_int
        return u_fd


    def generate_dataset(self, n_samples: int,
                         seed: int = None) -> np.ndarray:
        """
        Generuje (n_samples, n_points) macierz dokładnych rozwiązań u(x).
        """
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        solutions = []
        for _ in range(n_samples):
            _, u = self.analytical_solution()
            solutions.append(u)
        return np.array(solutions, dtype=np.float32)  # (N, 128)

    # ---------------------------------------------------------------- #
    # Normalizacja / denormalizacja                                     #
    # ---------------------------------------------------------------- #
    @staticmethod
    def normalize(u: np.ndarray) -> tuple:
        """Per-sample normalizacja do zakresu [-1, 1]."""
        scale = np.abs(u).max(axis=-1, keepdims=True) + 1e-8
        return (u / scale).astype(np.float32), scale

    @staticmethod
    def denormalize(u_norm: np.ndarray, scale: np.ndarray) -> np.ndarray:
        return u_norm * scale



class PoissonDDPMTrainer:
    """
    Trenuje DDPM1D z architekturą Conv1D (najlepsza dla SDEdit wg Eksp. II)
    na rozkładzie rozwiązań 1D równania Poissona.
    """

    def __init__(self, device=DEVICE, seed: int = 42):
        self.device  = device
        self.seed    = seed
        self.gen     = PoissonBVPGenerator(seed=seed)
        torch.manual_seed(seed)
        np.random.seed(seed)

    def _to_3d(self, arr: np.ndarray) -> torch.Tensor:
        return torch.from_numpy(arr).float().unsqueeze(1)

    def train(self,
              n_train:    int   = N_TRAIN,
              n_val:      int   = N_VAL,
              n_T:        int   = SDEDIT_T,
              schedule:   str   = SDEDIT_SCHEDULE,
              lr:         float = SDEDIT_LR,
              batch_size: int   = SDEDIT_BATCH,
              max_epochs: int   = SDEDIT_EPOCHS,
              patience:   int   = SDEDIT_PATIENCE,
              save_path:  str   = None) -> tuple:
        """
        Trenuje DDPM; zwraca (ddpm, history_dict).

        Strategia:
          • Dane: losowe rozwiązania analityczne Poissona (n_train sztuk)
          • Normalizacja per-sample do [-1,1] przed podaniem do sieci
          • AdamW z cosine-annealing + gradient clipping (max_norm=1.0)
          • Early-stopping z cierpliwością `patience` epok
        """
        print(f"\n{''*60}")
        print(f"  [DDPM] Trening Conv1D | T={n_T} | {schedule} | "
              f"lr={lr:.0e} | bs={batch_size}")
        print(f"  Dane: {n_train} treningowe + {n_val} walidacyjne")
        print(f"{''*60}")

        y_train = self.gen.generate_dataset(n_train)
        y_val   = self.gen.generate_dataset(n_val)
        y_train_n, _ = PoissonBVPGenerator.normalize(y_train)
        y_val_n,   _ = PoissonBVPGenerator.normalize(y_val)

        train_dl = DataLoader(
            TensorDataset(self._to_3d(y_train_n)),
            batch_size=batch_size, shuffle=True, num_workers=0)
        val_dl = DataLoader(
            TensorDataset(self._to_3d(y_val_n)),
            batch_size=batch_size, shuffle=False, num_workers=0)

        if _MODULES_OK:
            net   = DenoiseNet1D_Conv(
                data_dim=N_POINTS, time_emb_dim=64, base_channels=64
            ).to(self.device)
        else:
            raise RuntimeError("Moduły projektu niedostępne – nie można trenować.")

        betas  = get_beta_schedule(schedule, 1e-4, 0.02, n_T)
        ddpm   = DDPM1D(net, betas, n_T, self.device)
        optim  = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-3)
        sched  = torch.optim.lr_scheduler.CosineAnnealingLR(
                     optim, T_max=max_epochs)
        scaler = torch.cuda.amp.GradScaler(
                     enabled=(self.device.type == 'cuda'))

        best_val   = float('inf')
        best_state = None
        patience_ctr = 0
        history    = {'train': [], 'val': []}

        bar = tqdm(range(1, max_epochs + 1),
                   desc="Trening DDPM (Poisson)", ncols=80)
        for epoch in bar:
            net.train()
            e_loss = 0.0
            for (x,) in train_dl:
                x = x.to(self.device)
                optim.zero_grad(set_to_none=True)
                with torch.autocast(device_type=self.device.type,
                                    dtype=AMP_DTYPE):
                    loss = ddpm.compute_loss(x)
                scaler.scale(loss).backward()
                scaler.unscale_(optim)
                nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                scaler.step(optim)
                scaler.update()
                e_loss += loss.item()

            avg_train = e_loss / len(train_dl)
            history['train'].append(avg_train)
            sched.step()

            # Walidacja co 10 epok
            if epoch % 10 == 0:
                net.eval()
                v_loss = 0.0
                with torch.no_grad():
                    with torch.autocast(device_type=self.device.type,
                                        dtype=AMP_DTYPE):
                        for (vx,) in val_dl:
                            vx = vx.to(self.device)
                            v_loss += ddpm.compute_loss(vx).item() * vx.size(0)
                v_loss /= len(y_val)
                history['val'].append(v_loss)

                if v_loss < best_val:
                    best_val     = v_loss
                    best_state   = {k: v.cpu().clone()
                                    for k, v in net.state_dict().items()}
                    patience_ctr = 0
                else:
                    patience_ctr += 1

                bar.set_postfix({'train': f'{avg_train:.4f}',
                                 'val':   f'{v_loss:.4f}',
                                 'pat':   patience_ctr})

                if patience_ctr >= (patience // 10):
                    bar.write(f"  [Stop] early-stopping @ epoka {epoch}")
                    break
        bar.close()

        if best_state:
            net.load_state_dict(best_state)

        print(f"  Najlepszy val_loss = {best_val:.5f}")

        if save_path:
            torch.save({'model_state_dict': best_state,
                        'history': history,
                        'config': {'n_T': n_T, 'schedule': schedule}},
                       save_path)
            print(f"  Model zapisany → {save_path}")

        return ddpm, history


# ════════════════════════════════════════════════════════════════════
# 4.  EKSPERYMENT A – SDEdit JAKO KOREKTOR MRS + szum
# ════════════════════════════════════════════════════════════════════

class SDEditPoissonSolver:
    """
    Używa wytrenowanego DDPM do poprawiania zaburzonych rozwiązań MRS.

    Pipeline per-próbka:
      1.  Generuj u_fd = rozwiązanie MRS z szumem RHS (O(h²) + szum_std)
      2.  Normalizuj u_fd per-sample → [-1,1]
      3.  Forward diffusion do t_start = round(n_T * t_start_ratio):
              x_t = √ᾱ_t · u_fd_norm + √(1−ᾱ_t) · ε,  ε~N(0,I)
      4.  DDIM denoising (deterministic, skip_steps) → u_corrected_norm
      5.  Denormalizuj → u_corrected w oryginalnej skali

    Intuicja: DDPM nauczył się manifoldu czystych rozwiązań Poissona.
    SDEdit „projektuje" zaburzone rozwiązanie MRS na ten manifold.
    """

    def __init__(self, ddpm: 'DDPM1D', device=DEVICE,
                 t_ratio:    float = SDEDIT_T_RATIO,
                 skip_steps: int   = SDEDIT_SKIP):
        self.ddpm       = ddpm
        self.device     = device
        self.t_ratio    = t_ratio
        self.skip_steps = skip_steps
        self.gen        = PoissonBVPGenerator(seed=99)

    @torch.no_grad()
    def correct(self, u_fd: np.ndarray) -> np.ndarray:
        """Koryguje jedno rozwiązanie MRS; zwraca ndarray (n_points,)."""
        u_norm, scale = PoissonBVPGenerator.normalize(u_fd[None])  
        u_t = torch.from_numpy(u_norm).float().unsqueeze(1).to(self.device)
        # u_t: (1, 1, 128)

        t_start = max(1, round(self.ddpm.n_T * self.t_ratio))
        t_idx   = torch.tensor([t_start], device=self.device, dtype=torch.long)

        # Forward diffusion: q(x_t | x_0)
        a_bar   = self.ddpm.alphas_bar[t_idx].view(1, 1, 1)
        noise   = torch.randn_like(u_t)
        x_noisy = torch.sqrt(a_bar) * u_t + torch.sqrt(1 - a_bar) * noise

        # DDIM denoise
        u_out = self.ddpm.ddim_denoise_signal(
            x_noisy, t_start=t_start, skip_steps=self.skip_steps
        )  

        u_out_np = u_out.squeeze().cpu().numpy()           

        u_corrected = PoissonBVPGenerator.denormalize(u_out_np[None], scale)[0]
        
        u_corrected[0] = 0.0
        u_corrected[-1] = 0.0
        
        return u_corrected

    def run_benchmark(self, n_test: int = N_TEST,
                      noise_std: float = 0.5) -> dict:
        """
        Porównuje MRS bazowe vs SDEdit na n_test próbkach.

        noise_std: odch. std. szumu dodawanego do RHS równania MRS.
                   Wartość 0.5 odpowiada ~50% amplitudy typowej f(x).
        """
        mse_fd    = []
        mse_sd    = []
        mae_fd    = []
        mae_sd    = []
        l2_fd     = []
        l2_sd     = []
        t_exec    = []

        for _ in tqdm(range(n_test), desc="SDEdit benchmark", ncols=70):

            f, u_exact = self.gen.analytical_solution()
            u_fd_clean = self.gen.fd_solution(f, noise_std=0.0) 
            
            u_fd = u_fd_clean + noise_std * np.random.standard_normal(self.gen.n_points)
            u_fd[0] = u_fd[-1] = (0.0)


            t0 = time.time()
            u_corrected = self.correct(u_fd)
            t_exec.append(time.time() - t0)

            norm_ref = np.linalg.norm(u_exact) + 1e-8

            mse_fd.append(float(np.mean((u_exact - u_fd)** 2)))
            mse_sd.append(float(np.mean((u_exact - u_corrected)** 2)))
            mae_fd.append(float(np.mean(np.abs(u_exact - u_fd))))
            mae_sd.append(float(np.mean(np.abs(u_exact - u_corrected))))
            l2_fd.append( float(np.linalg.norm(u_exact - u_fd)/ norm_ref))
            l2_sd.append( float(np.linalg.norm(u_exact - u_corrected)/ norm_ref))

        improve_mse = 100.0 * (np.mean(mse_fd) - np.mean(mse_sd)) / (np.mean(mse_fd) + 1e-10)
        improve_l2  = 100.0 * (np.mean(l2_fd)  - np.mean(l2_sd))  / (np.mean(l2_fd)  + 1e-10)

        metrics = {
            'MSE_FD':       float(np.mean(mse_fd)),
            'MSE_SDEdit':   float(np.mean(mse_sd)),
            'MAE_FD':       float(np.mean(mae_fd)),
            'MAE_SDEdit':   float(np.mean(mae_sd)),
            'L2_FD_pct':    float(np.mean(l2_fd)* 100),
            'L2_SDEdit_pct':float(np.mean(l2_sd)* 100),
            'improve_MSE_pct': float(improve_mse),
            'improve_L2_pct':  float(improve_l2),
            'mean_exec_ms': float(np.mean(t_exec)* 1000),
            'noise_std':    noise_std,
            'n_test':       n_test,
        }
        return metrics

    def get_showcase_samples(self, n: int = 3,
                                 noise_std: float = 0.02) -> list:
            """Generuje n przykładowych przypadków do wizualizacji."""
            cases = []
            for _ in range(n):
                f, u_exact = self.gen.analytical_solution()
                
                u_fd_clean = self.gen.fd_solution(f, noise_std=0.0)  # Czyste rozw. MRS
                
                # szum bezpośrednio na gotowy sygnał
                u_fd = u_fd_clean + noise_std * np.random.standard_normal(self.gen.n_points)
                u_fd[0] = u_fd[-1] = 0.0  # Wymuszenie warunków brzegowych
    
                t0         = time.time()
                u_corrected = self.correct(u_fd)
    
                norm_ref = np.linalg.norm(u_exact) + 1e-8
    
                cases.append({
                    'x':           self.gen.x,
                    'f':           f,
                    'u_exact':     u_exact,
                    'u_fd':        u_fd,
                    'u_corrected': u_corrected,
                    'mse_fd':      float(np.mean((u_exact - u_fd)        ** 2)),
                    'mse_sd':      float(np.mean((u_exact - u_corrected) ** 2)),
                    'l2_fd':       float(np.linalg.norm(u_exact - u_fd)        / norm_ref),
                    'l2_sd':       float(np.linalg.norm(u_exact - u_corrected) / norm_ref),
                })
            return cases


# ════════════════════════════════════════════════════════════════════
# 5.  EKSPERYMENT B – FunDPS JAKO SOLVER ODWROTNY
# ════════════════════════════════════════════════════════════════════

class FunDPSPoissonSolver:
    """
    Używa FunDPS do rekonstrukcji pełnego rozwiązania u(x) z ~10% pomiarów.

    Interpretacja fizyczna:
      Problem odwrotny: czujniki rejestrują u(x) w 13/128 punktach.
      Pytanie: jakie jest pełne pole u(x)?

    Algorytm:
      1. Wytrenuj bezwarunkowy prior EDM na (pojedynczej) funkcji u(x)
         metodą EDM/score-matching z logarytmicznie rozłożonymi σ.
      2. Utwórz ForwardOperator(mask_idx): A · u = obs (selekcja 13 pkt.)
      3. Uruchom FunDPSSampler.sample z gradient-guidance DPS:
            â_{t-1} = [krok dyfuzji odwrotnej] − ζ·σ_t · ∇_{â₀} ‖A(â₀)−obs‖²
      4. Zwróć zrekonstruowane u(x).

    Uwaga: Trening priora na jednym przypadku (jak oryginalna implementacja)
    odpowiada podejściu „one-shot" - model overfittuje do kształtu u,
    co jest zamierzone w kontekście zadania interpolacji.
    """

    def __init__(self,
                 device      = DEVICE,
                 zeta:  float = FUNDPS_ZETA,
                 steps: int   = FUNDPS_STEPS,
                 epochs:int   = FUNDPS_PRIOR_EPOCHS,
                 obs_ratio: float = FUNDPS_OBS_RATIO,
                 lr: float    = FUNDPS_LR):
                    self.device    = device
                    self.zeta      = zeta
                    self.steps     = steps
                    self.epochs    = epochs
                    self.obs_ratio = obs_ratio
                    self.lr        = lr
                    self.gen       = PoissonBVPGenerator(seed=77)


    def _train_prior(self, u_tensor: torch.Tensor) -> tuple:
        """
        Trenuje prior EDM na pojedynczej funkcji u(x).
        u_tensor : (1, N_POINTS) float32

        Zwraca (model, loss_history).
        """
        if not _MODULES_OK:
            raise RuntimeError("Moduły projektu niedostępne.")

        model     = EDMDenoiser1D(data_dim=N_POINTS).to(self.device)
        optimizer = torch.optim.Adam(model.parameters(), lr=self.lr)
        batch_sz  = 32
        loss_hist = []

        model.train()
        for _ in range(self.epochs):
            optimizer.zero_grad()

            rnd   = torch.randn(batch_sz, device=self.device)
            sigma = (rnd * 1.2 - 1.2).exp()              # (bs,)

            # Biały szum (z Eksp. III)
            noise  = torch.randn(batch_sz, N_POINTS, device=self.device)
            noise *= sigma.unsqueeze(1)                    

            u_rep  = u_tensor.repeat(batch_sz, 1)         
            u_noisy = u_rep + noise

            u_pred = model(u_noisy, sigma)                

            # Ważona strata EDM (Karras et al., 2022)
            weight = (sigma ** 2 + 1) / (sigma ** 2 + 1e-5)
            loss   = (weight.unsqueeze(1) * (u_pred - u_rep) ** 2).mean()
            loss.backward()
            optimizer.step()
            loss_hist.append(loss.item())

        return model, loss_hist


    def reconstruct(self, u_exact: np.ndarray) -> dict:
        """
        Przyjmuje pełne u(x) tylko do symulacji obserwacji (ground truth).
        Zwraca słownik z u_rec, mask_idx, metrykami.
        """
        # rzadkie indeksy: BC (0, N-1) + losowe punkty wewnętrzne
        n_obs     = max(4, round(N_POINTS * self.obs_ratio))
        n_int     = n_obs - 2                  # bez krańców
        interior  = np.sort(
            np.random.choice(np.arange(1, N_POINTS - 1), n_int, replace=False)
        )
        mask_idx  = np.concatenate([[0], interior, [N_POINTS - 1]])

        # znane wartości u(x_obs)
        obs_tensor = torch.from_numpy(
            u_exact[mask_idx]
        ).float().unsqueeze(0).to(self.device)               

        # Operator pomiarowy A·u → u[mask_idx]
        fwd_op    = ForwardOperator(mask_idx)

        # Prior EDM na tej konkretnej funkcji
        u_t       = torch.from_numpy(u_exact).float().unsqueeze(0).to(self.device)
        t0_train  = time.time()
        model, loss_hist = self._train_prior(u_t)
        train_time = time.time() - t0_train

        # FunDPS sampling
        sampler   = FunDPSSampler(model, self.device)
        t0_inf    = time.time()
        u_rec_t   = sampler.sample(
            obs_tensor, fwd_op,
            num_steps=self.steps,
            zeta=self.zeta,
            data_dim=N_POINTS
        )
        inf_time = time.time() - t0_inf

        u_rec = u_rec_t.detach().cpu().numpy()[0]           
        norm_ref = np.linalg.norm(u_exact) + 1e-8

        return {
            'u_exact':    u_exact,
            'u_rec':      u_rec,
            'mask_idx':   mask_idx,
            'obs_vals':   u_exact[mask_idx],
            'x':          self.gen.x,
            'mse':        float(np.mean((u_exact - u_rec) ** 2)),
            'mae':        float(np.mean(np.abs(u_exact - u_rec))),
            'l2_pct':     float(np.linalg.norm(u_exact - u_rec) / norm_ref * 100),
            'train_time': train_time,
            'inf_time':   inf_time,
            'loss_hist':  loss_hist,
        }

    def run_cases(self, n_cases: int = 5) -> list:
        """Przeprowadza pełny Eksperyment B dla n_cases losowych BVP."""
        results = []
        print(f"\n{''*60}")
        print(f"  [FunDPS] Rekonstrukcja z {int(self.obs_ratio*100)}% "
              f"obserwacji (n_cases={n_cases})")
        print(f"  ζ={self.zeta} | steps={self.steps} | "
              f"epochs_prior={self.epochs}")
        print(f"{''*60}")

        for i in range(n_cases):
            f, u_exact = self.gen.analytical_solution()
            print(f"  Przypadek {i+1}/{n_cases} | "
                  f"u_max={np.abs(u_exact).max():.3f}")

            result = self.reconstruct(u_exact)
            results.append(result)
            print(f"    MSE={result['mse']:.6f} | "
                  f"L₂={result['l2_pct']:.2f}% | "
                  f"trening={result['train_time']:.0f}s | "
                  f"inferencja={result['inf_time']:.1f}s")

        return results




[OK] Moduły projektu wczytane pomyślnie z katalogu głównego.
[OK] Moduły projektu wczytane poprawnie.
[INFO] Wyniki będą zapisane w: experiments/de_application/
[INFO] Urządzenie obliczeniowe: cuda



In [2]:
# ════════════════════════════════════════════════════════════════════
# 6.  WIZUALIZACJA
# ════════════════════════════════════════════════════════════════════

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PALETTE          = ['#5ec962', '#fde725', '#3b528b']   
FONT_SIZE        = 10
TICK_SIZE        = 9
LEGEND_SIZE      = 9
ANNOT_SIZE       = 9
LINE_WIDTH       = 1.6
MARKER_SIZE      = 6
FIG_W            = 12 / 2.54   
FIG_H            =  8 / 2.54   

LEGEND_OUTSIDE = dict(
    loc='upper left',
    bbox_to_anchor=(1.02, 1.0),
    borderaxespad=0.,
    frameon=True,
    facecolor='white',
    edgecolor='black',
)

sns.set_style("whitegrid")          

plt.rcParams.update({
    'font.family':           'serif',
    'font.serif':            ['Times New Roman'],
    'text.color':            'black',
    'axes.labelcolor':       'black',
    'axes.edgecolor':        'black',
    'xtick.color':           'black',
    'ytick.color':           'black',
    'font.size':             FONT_SIZE,
    'axes.titlesize':        FONT_SIZE,
    'axes.titleweight':      'bold',
    'axes.labelsize':        FONT_SIZE,
    'xtick.labelsize':       TICK_SIZE,
    'ytick.labelsize':       TICK_SIZE,
    'legend.fontsize':       LEGEND_SIZE,
    'legend.title_fontsize': LEGEND_SIZE,
    'lines.linewidth':       LINE_WIDTH,
    'axes.linewidth':        0.8,
    'grid.linewidth':        0.5,
    'grid.linestyle':        '--',
    'grid.alpha':            0.5,
    'figure.dpi':            300,
    'savefig.dpi':           300,
    'figure.figsize':        [FIG_W, FIG_H],
    'figure.autolayout':     True,
})

_LEG_KW = dict(frameon=True, fontsize=LEGEND_SIZE, facecolor='white', edgecolor='black')


# ════════════════════════════════════════════════════════════════════
# SDEdit
# ════════════════════════════════════════════════════════════════════

def plot_sdedit(cases: list, metrics: dict, save_path: str = None):
    """Generuje osobne wykresy dla każdego przypadku SDEdit.

    Nazwa pliku pełni rolę tytułu - osie pozbawione nagłówka.
    """
    base_dir = os.path.dirname(save_path) if save_path else OUT_DIR
    stats = []
    for i, c in enumerate(cases):
        x = c['x']

        mse_fd  = np.mean((c['u_exact'] - c['u_fd'])        ** 2)
        mse_sd  = np.mean((c['u_exact'] - c['u_corrected']) ** 2)
        l2_pct  = (np.linalg.norm(c['u_exact'] - c['u_corrected'])
                   / np.linalg.norm(c['u_exact'])) * 100
        poprawa = (mse_fd - mse_sd) / mse_fd * 100
        stats.append((mse_fd, mse_sd, l2_pct, poprawa))

        print(
            f"[SDEdit | przypadek {i+1}]  "
            f"MSE: {mse_fd:.8f} → {mse_sd:.8f}  "
            f"(poprawa {poprawa:+.4f}%)  |  "
            f"L2 = {l2_pct:.4f}%"
        )


        fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
        ax.plot(x, c['u_exact'],     color=PALETTE[2], lw=LINE_WIDTH,
                label='Rozwiązanie')
        ax.plot(x, c['u_fd'],        color=PALETTE[1], lw=LINE_WIDTH, ls='--',
                label='MRS + szum')
        ax.plot(x, c['u_corrected'], color=PALETTE[0], lw=LINE_WIDTH,
                label='SDEdit')
        ax.set_xlabel('x')
        ax.set_ylabel('Wartość pola potencjału ($u$)')
        ax.legend(loc='upper right', **_LEG_KW)
        ax.set_axisbelow(True)
        plt.tight_layout()
        if save_path:
            plt.savefig(
                os.path.join(base_dir, f'sdedit_przypadek_{i+1}_sygnal.png'),
                dpi=300, bbox_inches='tight',
            )
        plt.close()

        fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
        ax.semilogy(
            x, np.abs(c['u_exact'] - c['u_fd']) + 1e-10,
            color=PALETTE[1], lw=LINE_WIDTH, ls='--',
            label=r'MRS + szum',
        )
        ax.semilogy(
            x, np.abs(c['u_exact'] - c['u_corrected']) + 1e-10,
            color=PALETTE[0], lw=LINE_WIDTH, 
            label=r'SDEdit',
        )
        ax.set_xlabel('x')
        ax.set_ylabel(r'Błąd bezwzględny $\Delta u$')
        ax.legend(loc='upper right', **_LEG_KW)
        ax.set_axisbelow(True)
        plt.tight_layout()
        if save_path:
            plt.savefig(
                os.path.join(base_dir, f'sdedit_przypadek_{i+1}_bledy.png'),
                dpi=300, bbox_inches='tight',
            )
        plt.close()
    print(
    f"\n[SDEdit | ŚREDNIA]  "
    f"poprawa MSE: {np.mean([s[3] for s in stats]):+.8f}%  |  "
    f"L2 śr. = {np.mean([s[2] for s in stats]):.8f}%"
    )

# ════════════════════════════════════════════════════════════════════
# FunDPS
# ════════════════════════════════════════════════════════════════════

def plot_fundps(results: list, save_path: str = None):
    """Generuje osobne wykresy dla każdego przypadku FunDPS.
    """
    base_dir = os.path.dirname(save_path) if save_path else OUT_DIR

    for i, r in enumerate(results):
        x    = r['x']
        midx = r['mask_idx']

        #  Odtworzenie pola u(x) z rzadkich obserwacji 
        fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
        ax.plot(x, r['u_exact'], color=PALETTE[2], lw=LINE_WIDTH,
                label='Rozwiązanie')
        ax.scatter(
            x[midx], r['obs_vals'],
            s=MARKER_SIZE ** 2, color=PALETTE[1],
            edgecolor='black', linewidths=0.6, zorder=5,
            label='Obserwacje',
        )
        ax.plot(x, r['u_rec'], color=PALETTE[0], lw=LINE_WIDTH, ls='--',
                label='FunDPS')
        ax.set_xlabel('x')
        ax.set_ylabel('Wartość pola potencjału ($u$)')
        ax.legend(loc='upper right', **_LEG_KW)
        ax.set_axisbelow(True)
        plt.tight_layout()
        if save_path:
            plt.savefig(
                os.path.join(base_dir, f'fundps_przypadek_{i+1}_rekonstrukcja.png'),
                dpi=300, bbox_inches='tight',
            )
        plt.close()

        fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
        lh = np.asarray(r['loss_hist'])
        ax.semilogy(lh, color=PALETTE[1], lw=LINE_WIDTH, label='Ważona strata EDM')
        ax.set_xlabel('Epoka')
        ax.set_ylabel('MSE')
        ax.legend(loc='upper right', **_LEG_KW)
        ax.set_axisbelow(True)
        plt.tight_layout()
        if save_path:
            plt.savefig(
                os.path.join(base_dir, f'fundps_przypadek_{i+1}_strata_EDM.png'),
                dpi=300, bbox_inches='tight',
            )
        plt.close()


# ════════════════════════════════════════════════════════════════════
# Historia uczenia DDPM
# ════════════════════════════════════════════════════════════════════

def plot_training_history(history: dict, save_path: str = None):
    """Wykres historii uczenia bazowego modelu DDPM.
    """
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    ax.semilogy(history['train'], lw=LINE_WIDTH, color=PALETTE[2],
                label='Błąd treningowy')
    if history['val']:
        val_x = [10 * k for k in range(len(history['val']))]
        ax.semilogy(val_x, history['val'], lw=LINE_WIDTH, color=PALETTE[0],
                    label='Błąd walidacyjny')
    ax.set_xlabel('Epoka')
    ax.set_ylabel('MSE')
    ax.legend(loc='upper right', **_LEG_KW)
    ax.set_axisbelow(True)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


# ════════════════════════════════════════════════════════════════════
# Panel porównawczy 
# ════════════════════════════════════════════════════════════════════

def plot_comparison_panel(sdedit_m: dict, fundps_r: list, save_path: str = None):
    """Rozbija panel zbiorczy na trzy samodzielne wykresy słupkowe.
    """
    base_dir = os.path.dirname(save_path) if save_path else OUT_DIR

    # Wykres 1: Porównanie MSE dla SDEdit
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    methods = ['MRS + szum\n(wejście)', 'SDEdit\n(wyjście)']
    vals    = [sdedit_m['MSE_FD'], sdedit_m['MSE_SDEdit']]
    bars = ax.bar(
        methods, vals,
        color=[PALETTE[2], PALETTE[0]],
        edgecolor='black', linewidth=0.8, width=0.4,
    )
    ax.bar_label(bars, fmt='%.8f', padding=3, fontsize=ANNOT_SIZE)
    ax.set_ylabel('Średni MSE')
    ax.set_axisbelow(True)
    plt.tight_layout()
    if save_path:
        plt.savefig(
            os.path.join(base_dir, 'porownanie_sdedit_mse.png'),
            dpi=300, bbox_inches='tight',
        )
    plt.close()

    # Wykres 2: Błędy względne L2 dla FunDPS
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    l2_vals      = [r['l2_pct'] for r in fundps_r]
    cases_labels = [f'P{k+1}' for k in range(len(l2_vals))]
    ax.bar(
        cases_labels, l2_vals,
        color=PALETTE[2], edgecolor='black', linewidth=0.8, width=0.5,
    )
    ax.axhline(
        np.mean(l2_vals), color='black', ls='--', lw=1.2,
        label=f'Średnia = {np.mean(l2_vals):.8f} %',
    )
    ax.set_xlabel('Profil testowy równania Poissona')
    ax.set_ylabel(r'Błąd względny $L_2$ [%]')
    ax.legend(loc='upper right', **_LEG_KW)
    ax.set_axisbelow(True)
    plt.tight_layout()
    if save_path:
        plt.savefig(
            os.path.join(base_dir, 'porownanie_fundps_l2.png'),
            dpi=300, bbox_inches='tight',
        )
    plt.close()

    #  Wykres 3: Czasy inferencji (skala logarytmiczna) 
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    t_sd = sdedit_m['mean_exec_ms']
    t_fp = np.mean([r['inf_time'] for r in fundps_r]) * 1000  
    ax.bar(
        ['SDEdit', 'FunDPS'], [t_sd, t_fp],
        color=[PALETTE[2], PALETTE[0]],
        edgecolor='black', linewidth=0.8, width=0.4,
    )
    ax.set_ylabel('Czas operacji [ms] (log)')
    ax.set_yscale('log')
    ax.set_axisbelow(True)
    plt.tight_layout()
    if save_path:
        plt.savefig(
            os.path.join(base_dir, 'porownanie_czasy_obliczeniowe.png'),
            dpi=300, bbox_inches='tight',
        )
    plt.close()

In [3]:
def run_de_application():
    """
    Uruchamia pełen pipeline:
      A.  Trening DDPM (lub wczytanie z cache)
      B.  Eksperyment A – SDEdit benchmark ilościowy + wizualizacja
      C.  Eksperyment B – FunDPS 5 przypadków + wizualizacja
      D.  Zbiorczy panel porównawczy
      E.  Zapis wyników do pickle
    """
    if not _MODULES_OK:
        raise RuntimeError(
            "Moduły projektu (models/) niedostępne. "
            "Uruchom skrypt z katalogu głównego projektu."
        )

    print(f"\n{'='*65}")
    print("  MODELE DYFUZYJNE → ROZWIĄZYWANIE 1D RÓWNANIA POISSONA")
    print(f"{'='*65}")
    print(f"  Równanie: −u″(x) = Σ cₙ sin(nπx),  x∈[0,1],  u(0)=u(1)=0")
    print(f"  Dyskretyzacja: {N_POINTS} punktów")
    print(f"  Urządzenie: {DEVICE}\n")

    # A.  Trening DDPM (SDEdit)                                      
    ckpt_path = os.path.join(OUT_DIR, 'ddpm_conv1d_poisson.pth')
    trainer   = PoissonDDPMTrainer(device=DEVICE, seed=42)

    if os.path.exists(ckpt_path):
        print(f"[DDPM] Wczytywanie wytrenowanego modelu: {ckpt_path}")
        net  = DenoiseNet1D_Conv(data_dim=N_POINTS, base_channels=64).to(DEVICE)
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        net.load_state_dict(ckpt['model_state_dict'])
        betas = get_beta_schedule(SDEDIT_SCHEDULE, 1e-4, 0.02, SDEDIT_T)
        ddpm  = DDPM1D(net, betas, SDEDIT_T, DEVICE)
        history = ckpt.get('history', {'train': [], 'val': []})
    else:
        print("[DDPM] Trening od podstaw...")
        ddpm, history = trainer.train(save_path=ckpt_path)

    if history['train']:
        plot_training_history(
            history,
            save_path=os.path.join(OUT_DIR, 'ddpm_training_curves.png')
        )

    # B.  Eksperyment A – SDEdit benchmark                           
    print(f"\n{''*60}")
    print(f"  [Eksp. A] SDEdit – benchmark ilościowy (n={N_TEST})")
    print(f"{''*60}")

    sdedit_solver = SDEditPoissonSolver(
        ddpm, device=DEVICE,
        t_ratio=SDEDIT_T_RATIO,
        skip_steps=SDEDIT_SKIP
    )
    sdedit_metrics = sdedit_solver.run_benchmark(n_test=N_TEST, noise_std=0.02)

    print(f"\n   Wyniki Eksperymentu A ")
    print(f"    MSE (MRS + szum):  {sdedit_metrics['MSE_FD']:.8f}")
    print(f"    MSE (SDEdit):    {sdedit_metrics['MSE_SDEdit']:.8f}")
    print(f"    Poprawa MSE:     {sdedit_metrics['improve_MSE_pct']:.4f}%")
    print(f"    L₂ (MRS + szum):  {sdedit_metrics['L2_FD_pct']:.8f}%")
    print(f"    L₂ (SDEdit):    {sdedit_metrics['L2_SDEdit_pct']:.8f}%")
    print(f"    Poprawa L₂:      {sdedit_metrics['improve_L2_pct']:.4f}%")
    print(f"    Czas inferencji: {sdedit_metrics['mean_exec_ms']:.8f} ms/próbkę")

    # Wizualizacja – 3 przykładowe przypadki
    showcase = sdedit_solver.get_showcase_samples(n=3, noise_std=0.02)
    plot_sdedit(
        showcase, sdedit_metrics,
        save_path=os.path.join(OUT_DIR, 'sdedit_poisson_results.png')
    )

    # C.  Eksperyment B – FunDPS                                     
    fundps_solver  = FunDPSPoissonSolver(
        device=DEVICE,
        zeta=FUNDPS_ZETA,
        steps=FUNDPS_STEPS,
        epochs=FUNDPS_PRIOR_EPOCHS
    )
    fundps_results = fundps_solver.run_cases(n_cases=5)

    l2_mean = np.mean([r['l2_pct']   for r in fundps_results])
    t_mean  = np.mean([r['inf_time'] for r in fundps_results])
    print(f"\n   Wyniki Eksperymentu B ")
    print(f"    Śr. błąd L₂:         {l2_mean:.8f}%")
    print(f"    Śr. MSE:             "
          f"{np.mean([r['mse'] for r in fundps_results]):.8f}")
    print(f"    Śr. czas inferencji: {t_mean:.8f} s/przypadek")

    plot_fundps(
        fundps_results,
        save_path=os.path.join(OUT_DIR, 'fundps_poisson_results.png')
    )

    # D.  Panel porównawczy                                          
    plot_comparison_panel(
        sdedit_metrics, fundps_results,
        save_path=os.path.join(OUT_DIR, 'comparison_panel.png')
    )

    # E.  Zapis wyników                                              
    results_out = {
        'equation': '−u″(x) = f(x), u(0)=u(1)=0',
        'sdedit_config': {
            'T': SDEDIT_T, 'schedule': SDEDIT_SCHEDULE,
            't_ratio': SDEDIT_T_RATIO, 'skip': SDEDIT_SKIP
        },
        'fundps_config': {
            'zeta': FUNDPS_ZETA, 'steps': FUNDPS_STEPS,
            'prior_epochs': FUNDPS_PRIOR_EPOCHS,
            'obs_ratio': FUNDPS_OBS_RATIO
        },
        'sdedit_metrics':  sdedit_metrics,
        'fundps_results': [
            {k: v for k, v in r.items() if k not in ('u_exact', 'u_rec')}
            for r in fundps_results
        ],
        'showcase_cases': showcase,
    }
    pkl_path = os.path.join(OUT_DIR, 'de_application_results.pkl')
    with open(pkl_path, 'wb') as fp:
        pickle.dump(results_out, fp)
    print(f"\n  Wyniki zapisane → {pkl_path}")

    # F.  Podsumowanie terminala                                     
    print(f"\n{'='*65}")
    print("  PODSUMOWANIE KOŃCOWE")
    print(f"{'='*65}")
    print(f"  RÓWNANIE: −u″(x) = Σcₙ sin(nπx),  x∈[0,1], u(0)=u(1)=0\n")
    print(f"  ┌──────────────┬─────────────────────┬────────────────────┐")
    print(f"  │              │   SDEdit (Eksp. A)  │  FunDPS (Eksp. B) │")
    print(f"  ├──────────────┼─────────────────────┼────────────────────┤")
    print(f"  │ Cel          │ Korekcja MRS + szum   │ Imputacja 10% obs │")
    print(f"  │ MSE (śr.)    │ {sdedit_metrics['MSE_SDEdit']:^19.6f} │ "
          f"{np.mean([r['mse'] for r in fundps_results]):^18.6f} │")
    print(f"  │ L₂ err [%]   │ {sdedit_metrics['L2_SDEdit_pct']:^19.8f} │ "
          f"{l2_mean:^18.2f} │")
    print(f"  │ Czas [ms]    │ {sdedit_metrics['mean_exec_ms']:^19.8f} │ "
          f"{t_mean*1000:^18.8f} │")
    print(f"  └──────────────┴─────────────────────┴────────────────────┘")
    print(f"\n  Pliki wynikowe w: {OUT_DIR}/")
    print(f"{'='*65}\n")

    return results_out

In [4]:
if __name__ == '__main__':
    results = run_de_application()


  MODELE DYFUZYJNE → ROZWIĄZYWANIE 1D RÓWNANIA POISSONA
  Równanie: −u″(x) = Σ cₙ sin(nπx),  x∈[0,1],  u(0)=u(1)=0
  Dyskretyzacja: 128 punktów
  Urządzenie: cuda

[DDPM] Trening od podstaw...


  [DDPM] Trening Conv1D | T=80 | linear | lr=1e-03 | bs=128
  Dane: 5000 treningowe + 500 walidacyjne



Trening DDPM (Poisson):  12%| | 609/5000 [19:20<2:19:26,  1.91s/it, train=0.0222


  [Stop] early-stopping @ epoka 610
  Najlepszy val_loss = 0.02063
  Model zapisany → experiments/de_application\ddpm_conv1d_poisson.pth


  [Eksp. A] SDEdit – benchmark ilościowy (n=100)



SDEdit benchmark: 100%|█████████████| 100/100 [00:08<00:00, 11.41it/s]



   Wyniki Eksperymentu A 
    MSE (MRS + szum):  0.00039224
    MSE (SDEdit):    0.00006958
    Poprawa MSE:     82.2614%
    L₂ (MRS + szum):  70.47464887%
    L₂ (SDEdit):    23.47769427%
    Poprawa L₂:      66.6863%
    Czas inferencji: 85.64625740 ms/próbkę
[SDEdit | przypadek 1]  MSE: 0.00037090 → 0.00004608  (poprawa +87.5762%)  |  L2 = 29.0447%
[SDEdit | przypadek 2]  MSE: 0.00044354 → 0.00018566  (poprawa +58.1400%)  |  L2 = 10.2535%
[SDEdit | przypadek 3]  MSE: 0.00040386 → 0.00002438  (poprawa +93.9635%)  |  L2 = 16.9918%

[SDEdit | ŚREDNIA]  poprawa MSE: +79.89324661%  |  L2 śr. = 18.76334890%


  [FunDPS] Rekonstrukcja z 10% obserwacji (n_cases=5)
  ζ=2.0 | steps=5 | epochs_prior=2000

  Przypadek 1/5 | u_max=0.045
    MSE=0.000000 | L₂=1.13% | trening=16s | inferencja=0.1s
  Przypadek 2/5 | u_max=0.186
    MSE=0.000000 | L₂=0.44% | trening=16s | inferencja=0.0s
  Przypadek 3/5 | u_max=0.100
    MSE=0.000000 | L₂=1.01% | trening=16s | inferencja=0.0s
  Przypadek 4/5 | u_m